In [ ]:
from dataclasses import dataclass
from enum import Enum
from typing import List, Dict, Optional

class DiseaseType(Enum):
    INFECTIOUS = "infectious"
    NON_COMMUNICABLE = "non_communicable"

class TransmissionRoute(Enum):
    SEXUAL = "sexual"
    AIRBORNE = "airborne"
    VECTOR = "vector"
    VERTICAL = "vertical"
    NONE = "none"

@dataclass
class DiseaseDeclaration:
    """Complete declarative specification of a disease"""

    # Core characteristics
    name: str
    disease_type: DiseaseType
    transmission_routes: List[TransmissionRoute]

    # Disease progression
    stages: List[str]
    initial_stage: str
    terminal_stages: List[str]

    # Risk factors (all potential predictors)
    risk_factors: Dict[str, List[str]]  # category -> factor list

    # All possible health system tasks
    available_tasks: Dict[str, 'HealthSystemTask']

    # Linear model specifications
    risk_models: Dict[str, 'ModelSpec']
    progression_models: Dict[str, 'ModelSpec']

In [ ]:
@dataclass
class ModelSpec:
    """Specification for linear/survival models"""
    model_type: str  # 'linear', 'logistic', 'survival'
    predictors: List[str]
    interactions: List[str] = None
    time_varying: bool = False

@dataclass
class HealthSystemTask:
    """Health system interaction task"""
    task_id: str
    category: str  # 'prevention', 'screening', 'treatment', 'palliative'
    facility_level: str
    consumables: List[str]
    footprint: Dict[str, float]
    eligibility_criteria: List[str]

class PolicyConfiguration:
    """Policy settings determining which tasks are enabled"""

    def __init__(self, enabled_tasks: List[str] = None):
        self.enabled_tasks = enabled_tasks or []

    def is_task_enabled(self, task_id: str, year: int) -> bool:
        # Policy logic

In [ ]:
# HIV Disease Declaration
hiv_declaration = DiseaseDeclaration(
    name="hiv",
    disease_type=DiseaseType.INFECTIOUS,
    transmission_routes=[TransmissionRoute.SEXUAL, TransmissionRoute.VERTICAL],

    # Disease stages
    stages=["susceptible", "acute_hiv", "chronic_hiv", "aids", "hiv_death"],
    initial_stage="susceptible",
    terminal_stages=["hiv_death"],

    # Risk factors by category
    risk_factors={
        "demographic": ["age_exact_years", "sex", "district"],
        "socioeconomic": ["wealth_quintile", "education_level"],
        "behavioral": ["sexual_partners", "condom_use", "circumcised"],
        "clinical": ["other_sti", "tb_status"],
        "program": ["hiv_tested", "art_status", "viral_suppressed"]
    },

    # All possible health system tasks
    available_tasks={
        "hiv_test": HealthSystemTask(
            task_id="hiv_test",
            category="screening",
            facility_level="1a",
            consumables=["hiv_test_kit"],
            footprint={"nurse_time": 0.25},
            eligibility_criteria=["age >= 15", "not hiv_diagnosed"]
        ),
        "hiv_self_test": HealthSystemTask(
            task_id="hiv_self_test",
            category="screening",
            facility_level="community",
            consumables=["self_test_kit"],
            footprint={},
            eligibility_criteria=["age >= 15"]
        ),
        "art_initiation": HealthSystemTask(
            task_id="art_initiation",
            category="treatment",
            facility_level="1a",
            consumables=["art_drugs", "cd4_test", "viral_load_test"],
            footprint={"clinician_time": 1.0},
            eligibility_criteria=["hiv_diagnosed", "not art_status"]
        ),
        "art_continuation": HealthSystemTask(
            task_id="art_continuation",
            category="treatment",
            facility_level="1a",
            consumables=["art_drugs"],
            footprint={"nurse_time": 0.5},
            eligibility_criteria=["art_status"]
        ),
        "prep_provision": HealthSystemTask(
            task_id="prep_provision",
            category="prevention",
            facility_level="1a",
            consumables=["prep_drugs", "hiv_test_kit"],
            footprint={"clinician_time": 0.75},
            eligibility_criteria=["high_risk", "hiv_negative"]
        ),
        "circumcision": HealthSystemTask(
            task_id="circumcision",
            category="prevention",
            facility_level="1b",
            consumables=["surgical_kit"],
            footprint={"surgeon_time": 1.5},
            eligibility_criteria=["sex == 'M'", "not circumcised", "age >= 15"]
        )
    },

    # Risk and progression models
    risk_models={
        "infection_risk": ModelSpec(
            model_type="logistic",
            predictors=["age_exact_years", "sex", "sexual_partners", "condom_use", "circumcised", "other_sti"],
            interactions=["age_exact_years:sex", "sexual_partners:condom_use"]
        ),
        "transmission_probability": ModelSpec(
            model_type="logistic",
            predictors=["viral_load", "other_sti", "circumcised"],
            interactions=["viral_load:other_sti"]
        )
    },

    progression_models={
        "time_to_aids": ModelSpec(
            model_type="survival",
            predictors=["age_at_infection", "cd4_count", "art_status"],
            interactions=["age_at_infection:cd4_count"]
        ),
        "aids_mortality": ModelSpec(
            model_type="survival",
            predictors=["age_exact_years", "cd4_count", "art_status", "tb_status"],
            interactions=["cd4_count:art_status"]
        )
    }
)

In [ ]:
# Cervical Cancer Disease Declaration
cervical_cancer_declaration = DiseaseDeclaration(
    name="cervical_cancer",
    disease_type=DiseaseType.NON_COMMUNICABLE,
    transmission_routes=[TransmissionRoute.NONE],  # Though HPV is infectious cause

    # Disease stages
    stages=["healthy", "hpv_infection", "cin1", "cin2_3", "invasive_cancer", "cancer_death"],
    initial_stage="healthy",
    terminal_stages=["cancer_death"],

    # Risk factors by category
    risk_factors={
        "demographic": ["age_exact_years", "district", "parity"],
        "socioeconomic": ["wealth_quintile", "education_level"],
        "behavioral": ["smoking", "sexual_partners", "age_first_sex"],
        "clinical": ["hiv_status", "hpv_type", "contraceptive_use"],
        "program": ["screened_ever", "screened_recent", "vaccinated_hpv"]
    },

    # All possible health system tasks
    available_tasks={
        "hpv_vaccination": HealthSystemTask(
            task_id="hpv_vaccination",
            category="prevention",
            facility_level="1a",
            consumables=["hpv_vaccine"],
            footprint={"nurse_time": 0.25},
            eligibility_criteria=["age >= 9", "age <= 26", "not hpv_vaccinated"]
        ),
        "via_screening": HealthSystemTask(
            task_id="via_screening",
            category="screening",
            facility_level="1a",
            consumables=["via_supplies"],
            footprint={"nurse_time": 0.5},
            eligibility_criteria=["age >= 25", "age <= 65", "sex == 'F'"]
        ),
        "hpv_test": HealthSystemTask(
            task_id="hpv_test",
            category="screening",
            facility_level="1b",
            consumables=["hpv_test_kit"],
            footprint={"technician_time": 0.25},
            eligibility_criteria=["age >= 30", "sex == 'F'"]
        ),
        "colposcopy": HealthSystemTask(
            task_id="colposcopy",
            category="screening",
            facility_level="2",
            consumables=["colposcopy_supplies"],
            footprint={"specialist_time": 1.0},
            eligibility_criteria=["abnormal_screening"]
        ),
        "cryotherapy": HealthSystemTask(
            task_id="cryotherapy",
            category="treatment",
            facility_level="1b",
            consumables=["cryotherapy_supplies"],
            footprint={"clinician_time": 0.75},
            eligibility_criteria=["cin1", "cin2_3"]
        ),
        "leep": HealthSystemTask(
            task_id="leep",
            category="treatment",
            facility_level="2",
            consumables=["leep_supplies"],
            footprint={"specialist_time": 1.5},
            eligibility_criteria=["cin2_3", "failed_cryotherapy"]
        ),
        "cancer_surgery": HealthSystemTask(
            task_id="cancer_surgery",
            category="treatment",
            facility_level="3",
            consumables=["surgery_supplies"],
            footprint={"surgeon_time": 4.0, "bed_days": 5},
            eligibility_criteria=["invasive_cancer", "early_stage"]
        ),
        "chemotherapy": HealthSystemTask(
            task_id="chemotherapy",
            category="treatment",
            facility_level="3",
            consumables=["chemo_drugs"],
            footprint={"oncologist_time": 1.0},
            eligibility_criteria=["invasive_cancer"]
        ),
        "palliative_care": HealthSystemTask(
            task_id="palliative_care",
            category="palliative",
            facility_level="1a",
            consumables=["pain_medication"],
            footprint={"nurse_time": 0.5},
            eligibility_criteria=["advanced_cancer"]
        )
    },

    # Risk and progression models
    risk_models={
        "hpv_infection_risk": ModelSpec(
            model_type="logistic",
            predictors=["age_exact_years", "sexual_partners", "age_first_sex", "hiv_status"],
            interactions=["age_exact_years:sexual_partners"]
        ),
        "screening_attendance": ModelSpec(
            model_type="logistic",
            predictors=["age_exact_years", "wealth_quintile", "education_level", "district"],
            interactions=["wealth_quintile:education_level"]
        )
    },

    progression_models={
        "hpv_clearance": ModelSpec(
            model_type="logistic",
            predictors=["age_exact_years", "hpv_type", "hiv_status", "smoking"],
            interactions=["hpv_type:hiv_status"]
        ),
        "progression_to_cin": ModelSpec(
            model_type="logistic",
            predictors=["hpv_type", "hiv_status", "smoking", "parity"],
            interactions=["hpv_type:hiv_status"]
        ),
        "cancer_progression": ModelSpec(
            model_type="survival",
            predictors=["age_at_diagnosis", "stage", "hiv_status"],
            interactions=["stage:hiv_status"]